# Comprehensive SFT Training Tutorial

This notebook provides a comprehensive guide to Supervised Fine-Tuning (SFT) using the Training Hub library. We cover:

- **All available parameters** and their detailed explanations
- **Single-node and multi-node training** configurations
- **Popular model examples** (Qwen 2.5 7B, Llama 3.1 8B, Phi 4 Mini, etc.)
- **Best practices and troubleshooting**

This tutorial serves as both a learning resource and a template you can adapt for your specific fine-tuning needs.

> For a minimal quickstart, see `sft_quickstart.py` in this directory.

## Setup and Imports

In [ ]:
from training_hub import sft

import os
import time
from datetime import datetime
from pathlib import Path

## Data Format Requirements

Training Hub expects data in **JSONL** format with a `messages` array per line:

```json
{"messages": [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": "Hello"}, {"role": "assistant", "content": "Hi there!"}]}
```

### Message Roles

| Role | Purpose |
|------|---------|
| `system` | System prompt (always masked from loss) |
| `user` | User turn |
| `assistant` | Model response (trained on by default) |
| `pretraining` | Raw text for pretraining-style data |

### Masking Behavior

By default, loss is computed only on `assistant` turns. Add `"unmask": true` to a sample to train on **all** content except system messages (useful for pretraining-style data):

```json
{"messages": [{"role": "user", "content": "The capital of France is"}, {"role": "assistant", "content": "Paris."}], "unmask": true}
```

## Model Configuration Examples

Choose a starting configuration that matches your model. Adjust based on your hardware and dataset.

In [ ]:
MODEL_CONFIGS = {
    "qwen_7b": {
        "model_path": "Qwen/Qwen2.5-7B-Instruct",
        "max_tokens_per_gpu": 20_000,
        "max_seq_len": 16_384,
        "effective_batch_size": 128,
        "learning_rate": 1e-5,
    },
    "llama_8b": {
        "model_path": "meta-llama/Llama-3.1-8B-Instruct",
        "max_tokens_per_gpu": 18_000,
        "max_seq_len": 16_384,
        "effective_batch_size": 128,
        "learning_rate": 1e-5,
    },
    "phi_mini": {
        "model_path": "microsoft/Phi-4-mini-instruct",
        "max_tokens_per_gpu": 25_000,
        "max_seq_len": 8_192,
        "effective_batch_size": 64,
        "learning_rate": 5e-6,
    },
    "small_1b_3b": {
        "model_path": "/path/to/small-model",
        "max_tokens_per_gpu": 40_000,
        "max_seq_len": 32_768,
        "effective_batch_size": 512,
        "learning_rate": 3e-5,
    },
}

# ---- Select your configuration ----
selected = MODEL_CONFIGS["qwen_7b"]

for k, v in selected.items():
    print(f"  {k}: {v}")

## Complete Parameter Configuration

Below is the full set of SFT parameters with explanations.

In [ ]:
experiment_name = "sft_comprehensive_example"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# ---- Required ----
model_path       = selected["model_path"]
data_path        = "/path/to/your/training_data.jsonl"
ckpt_output_dir  = f"/path/to/checkpoints/{experiment_name}_{timestamp}"

# ---- Core training ----
num_epochs           = 3
effective_batch_size = selected["effective_batch_size"]
learning_rate        = selected["learning_rate"]
max_seq_len          = selected["max_seq_len"]
max_tokens_per_gpu   = selected["max_tokens_per_gpu"]

# ---- Data processing ----
data_output_dir = "/dev/shm"   # RAM disk for fast data loading
warmup_steps    = 100

# ---- Checkpointing ----
save_samples                    = 0      # 0 = only checkpoint at epoch boundaries
checkpoint_at_epoch             = True
accelerate_full_state_at_epoch  = True   # enables automatic resume on restart

print(f"Experiment:  {experiment_name}_{timestamp}")
print(f"Model:       {model_path}")
print(f"Data:        {data_path}")
print(f"Output:      {ckpt_output_dir}")

## Distributed Training Configuration

Training Hub wraps `torchrun` internally. Configure the distributed backend here.

In [ ]:
DIST_PRESETS = {
    "single_gpu": dict(nproc_per_node=1, nnodes=1, node_rank=0,
                        rdzv_id=1, rdzv_endpoint="127.0.0.1:29500"),
    "single_node_8gpu": dict(nproc_per_node=8, nnodes=1, node_rank=0,
                              rdzv_id=100, rdzv_endpoint="127.0.0.1:29500"),
    "multi_node_master": dict(nproc_per_node=8, nnodes=4, node_rank=0,
                               rdzv_id=42, rdzv_endpoint="10.0.0.1:29500"),
    "multi_node_worker": dict(nproc_per_node=8, nnodes=4, node_rank=1,
                               rdzv_id=42, rdzv_endpoint="10.0.0.1:29500"),
}

# ---- Select your setup ----
dist = DIST_PRESETS["single_node_8gpu"]

total_gpus = dist["nproc_per_node"] * dist["nnodes"]
print(f"Total GPUs: {total_gpus}  ({dist['nproc_per_node']} per node x {dist['nnodes']} nodes)")

if dist["nnodes"] > 1:
    print(f"\nMulti-node checklist:")
    print(f"  - All nodes must reach {dist['rdzv_endpoint']}")
    print(f"  - Use the same rdzv_id ({dist['rdzv_id']}) everywhere")
    print(f"  - Set node_rank uniquely (0, 1, 2, ...)")
    print(f"  - Start all nodes simultaneously")

## Execute Training

Assemble all parameters and launch the training job.

In [ ]:
training_params = dict(
    # Required
    model_path=model_path,
    data_path=data_path,
    ckpt_output_dir=ckpt_output_dir,
    # Core training
    num_epochs=num_epochs,
    effective_batch_size=effective_batch_size,
    learning_rate=learning_rate,
    max_seq_len=max_seq_len,
    max_tokens_per_gpu=max_tokens_per_gpu,
    # Data processing
    data_output_dir=data_output_dir,
    warmup_steps=warmup_steps,
    save_samples=save_samples,
    # Checkpointing
    checkpoint_at_epoch=checkpoint_at_epoch,
    accelerate_full_state_at_epoch=accelerate_full_state_at_epoch,
    # Distributed
    **dist,
)

print("Training configuration:")
for k, v in training_params.items():
    print(f"  {k}: {v}")

start_time = time.time()

try:
    result = sft(**training_params)
    elapsed = time.time() - start_time
    print(f"\nTraining completed in {elapsed / 3600:.2f} hours")
    print(f"Checkpoints saved to: {ckpt_output_dir}")
except Exception as exc:
    elapsed = time.time() - start_time
    print(f"\nTraining failed after {elapsed / 60:.1f} minutes: {exc}")
    print("\nTroubleshooting checklist:")
    print("  - Verify model_path and data_path exist")
    print("  - Reduce max_tokens_per_gpu if you see OOM errors")
    print("  - Check network connectivity for multi-node setups")
    raise

## Post-Training Analysis

In [ ]:
hf_dir = os.path.join(ckpt_output_dir, "hf_format")

if os.path.isdir(hf_dir):
    checkpoints = sorted(
        d for d in os.listdir(hf_dir)
        if os.path.isdir(os.path.join(hf_dir, d))
    )
    print(f"Found {len(checkpoints)} checkpoint(s):")
    for ckpt in checkpoints:
        print(f"  {ckpt}")

    if checkpoints:
        final = os.path.join(hf_dir, checkpoints[-1])
        print(f"\nFinal checkpoint: {final}")
        print("\nLoad your fine-tuned model:")
        print(f"  from transformers import AutoModelForCausalLM, AutoTokenizer")
        print(f"  model = AutoModelForCausalLM.from_pretrained('{final}')")
        print(f"  tokenizer = AutoTokenizer.from_pretrained('{final}')")
else:
    print(f"Checkpoint directory not found: {hf_dir}")

## Parameter Reference

### Core Parameters

| Parameter | Required | Description |
|-----------|----------|-------------|
| `model_path` | Yes | HuggingFace model ID or local path |
| `data_path` | Yes | Path to JSONL training data |
| `ckpt_output_dir` | Yes | Directory for checkpoint output |
| `num_epochs` | No | Number of training epochs |
| `effective_batch_size` | No | Global batch size across all GPUs |
| `learning_rate` | No | Optimizer learning rate |
| `max_seq_len` | No | Maximum sequence length in tokens |
| `max_tokens_per_gpu` | No | Token budget per GPU per step (controls micro-batch packing and memory) |

### Checkpointing

| Parameter | Description |
|-----------|-------------|
| `checkpoint_at_epoch` | Save a checkpoint at each epoch boundary |
| `accelerate_full_state_at_epoch` | Save full optimizer state for automatic resume |
| `save_samples` | Checkpoint every N samples (0 = disabled) |

### Distributed Training

| Parameter | Description |
|-----------|-------------|
| `nproc_per_node` | GPUs per node |
| `nnodes` | Total number of nodes |
| `node_rank` | This node's rank (0 = master) |
| `rdzv_id` | Unique job ID for rendezvous |
| `rdzv_endpoint` | Master node address (`host:port`) |

### Recommended Configurations

| Model | `max_tokens_per_gpu` | `learning_rate` |
|-------|---------------------|-----------------|
| Qwen 2.5 7B | 20,000 | 1e-5 |
| Llama 3.1 8B | 18,000 | 1e-5 |
| Phi 4 Mini | 25,000 | 5e-6 |
| 1B–3B models | 40,000 | 3e-5 |